# 日本語のベンチマークで複数の LLM を同じ条件で比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_llm/llm_benchmark.ipynb)

NLP クックブック 12-09 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

大きさの違う LLM を、日本語のベンチマークで同じ条件のもとで比べたい。答えの取り出し方、プロンプト、few-shot の数で点数がどれくらい変わるか、評価の件数でどれくらいばらつくかも確かめたい。

In [1]:
!pip install -q datasets

In [2]:
import gc, re, time, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
# JGLUE (Kurihara et al., 2022) の JCommonsenseQA (5 択の常識問題) と JSQuAD (読解)。ライセンスは CC BY-SA 4.0
jcqa = load_dataset("sbintuitions/JCommonsenseQA", split="validation").shuffle(seed=0).select(range(500))
jsquad = load_dataset("sbintuitions/JSQuAD", split="validation").shuffle(seed=0).select(range(200))
jcqa_train = load_dataset("sbintuitions/JCommonsenseQA", split="train").shuffle(seed=0)    # few-shot の例は学習データから取る
jsquad_train = load_dataset("sbintuitions/JSQuAD", split="train").shuffle(seed=0).select(range(2))

# few-shot の例を 2 通り選ぶ: 先頭の 3 問そのまま (正解が偏っている) と、正解の番号が重ならない 3 問
skewed = [jcqa_train[i] for i in range(3)]
balanced = []
for ex in jcqa_train:
    if ex["label"] not in [e["label"] for e in balanced]:
        balanced.append(ex)
    if len(balanced) == 3:
        break
print("例の正解:", [e["label"] + 1 for e in skewed], [e["label"] + 1 for e in balanced])
print("評価データの正解の分布:", sorted(Counter(l + 1 for l in jcqa["label"]).items()))

def mc_prompt(ex):
    listing = "\n".join(f"{i + 1}. {ex[f'choice{i}']}" for i in range(5))
    return f"次の質問の答えとして最も適切なものを選び、番号だけを答えてください。\n\n質問: {ex['question']}\n{listing}"

def qa_prompt(ex):
    context = ex["context"].split("[SEP]")[-1].strip()      # 「記事タイトル [SEP] 段落」から段落だけを使う
    return f"次の文章を読んで、質問の答えを文章中からそのまま抜き出してください。答えだけを短く書いてください。\n\n文章: {context}\n\n質問: {ex['question']}"

def normalize(text):
    text = unicodedata.normalize("NFKC", text).lower()
    return re.sub(r"[\s。、．，,.!?！？「」『』()（）]", "", text)

def char_f1(pred, golds):
    """文字単位の F1 (正解が複数あれば最大値)。JSQuAD の標準的な指標"""
    best = 0.0
    for g in golds:
        common = sum((Counter(normalize(pred)) & Counter(normalize(g))).values())
        if common:
            p, r = common / len(normalize(pred)), common / len(normalize(g))
            best = max(best, 2 * p * r / (p + r))
    return best

print(mc_prompt(jcqa[0]))

例の正解: [4, 5, 5] [4, 5, 1]
評価データの正解の分布: [(1, 94), (2, 117), (3, 105), (4, 101), (5, 83)]
次の質問の答えとして最も適切なものを選び、番号だけを答えてください。

質問: 福井県の南西部、若狭のほぼ中央に位置する市は？
1. 小鍋
2. 小浜
3. 大浜
4. 小売
5. 高浜


In [3]:
import sentencepiece as spm
from huggingface_hub import hf_hub_download
from tokenizers import models

def load_tokenizer(name):
    tokenizer = AutoTokenizer.from_pretrained(name, padding_side="left")
    # transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
    # 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま。この章の「オープンな LLM とチャットする」を参照)
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
    tokenizer.backend_tokenizer.model = models.Unigram(
        [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def chat(prompt, shots=()):
    """few-shot の例は「ユーザーの問題 → アシスタントの答え」の対話として前に置く"""
    messages = []
    for p, a in shots:
        messages += [{"role": "user", "content": p}, {"role": "assistant", "content": a}]
    return tokenizer.apply_chat_template(messages + [{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)

def batches(texts, bs=32):
    for i in range(0, len(texts), bs):
        yield tokenizer(texts[i:i + bs], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)

@torch.no_grad()
def number_prob(texts):
    """方式 1: 次のトークンとして「1」〜「5」のどれの確率が高いか (生成はしない)"""
    digits = [tokenizer(str(i), add_special_tokens=False)["input_ids"][-1] for i in range(1, 6)]
    preds = []
    for enc in batches(texts):
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)          # 左パディングでも位置を 0 から数える
        logits = model(**enc, position_ids=pos, logits_to_keep=1).logits      # 最後の位置のロジットだけを計算する
        preds += logits[:, -1, digits].argmax(-1).tolist()
    return np.array(preds)

@torch.no_grad()
def generate(texts, max_new_tokens):
    """方式 2: 貪欲法で生成させ、あとで答えを取り出す"""
    outs = []
    for enc in batches(texts):
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        outs += tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

def parse_number(text):
    m = re.search(r"[1-5１-５]", text)                    # 最初に出てきた 1〜5 の数字を答えとみなす
    return int(unicodedata.normalize("NFKC", m.group())) - 1 if m else -1   # 見つからなければ不正解 (-1)

@torch.no_grad()
def choice_loglik(exs):
    """方式 3: 選択肢を並べずに質問だけを見せ、各選択肢の文字列を応答として続けたときの対数尤度の合計とトークン数"""
    seqs, spans = [], []
    for ex in exs:
        prompt_ids = tokenizer(chat(f"次の質問に答えてください。\n\n質問: {ex['question']}"), add_special_tokens=False)["input_ids"]
        for i in range(5):
            c = tokenizer(ex[f"choice{i}"], add_special_tokens=False)["input_ids"]
            seqs.append(prompt_ids + c); spans.append((len(prompt_ids), c))
    L = max(map(len, seqs))
    ids = torch.tensor([s + [tokenizer.pad_token_id] * (L - len(s)) for s in seqs], device=model.device)   # 右パディング
    logits = model(ids, attention_mask=(torch.arange(L, device=model.device) < torch.tensor([len(s) for s in seqs], device=model.device)[:, None]).long()).logits
    total = [logits[k, n - 1:n - 1 + len(c)].float().log_softmax(-1).gather(1, torch.tensor(c, device=model.device)[:, None]).sum().item()
             for k, (n, c) in enumerate(spans)]
    return np.array(total).reshape(-1, 5), np.array([len(c) for _, c in spans]).reshape(-1, 5)

In [4]:
# 3 つのモデルを順に読み込み、すべて同じ問題・同じプロンプト・同じ答えの取り出し方で評価する (Colab の T4 で合計 10〜15 分程度)
MODELS = ["sbintuitions/sarashina2.2-0.5b-instruct-v0.1", "sbintuitions/sarashina2.2-1b-instruct-v0.1", "sbintuitions/sarashina2.2-3b-instruct-v0.1"]
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
gold = np.array(jcqa["label"])
SHOTS = {"0-shot": [], "3-shot (正解が偏った例)": [(mc_prompt(e), str(e["label"] + 1)) for e in skewed],
         "3-shot (正解が重ならない例)": [(mc_prompt(e), str(e["label"] + 1)) for e in balanced]}
qa_shots = [(qa_prompt(e), e["answers"]["text"][0]) for e in jsquad_train]
per_item, outputs, preds = {}, {}, {}         # per_item[モデル, 設定] = 問題ごとの得点 (0/1 または F1)

for name in MODELS:
    t = time.time()
    tokenizer = load_tokenizer(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="cuda").eval()
    m = name.split("/")[-1].replace("-instruct-v0.1", "")

    scores, lens = map(np.concatenate, zip(*[choice_loglik(jcqa.select(range(i, min(i + 8, len(jcqa))))) for i in range(0, len(jcqa), 8)]))
    per_item[m, "JCQA 選択肢の尤度 (合計)"] = scores.argmax(1) == gold
    per_item[m, "JCQA 選択肢の尤度 (トークン平均)"] = (scores / lens).argmax(1) == gold
    for shot, examples in SHOTS.items():
        texts = [chat(mc_prompt(ex), examples) for ex in jcqa]
        preds[m, shot] = number_prob(texts)
        per_item[m, f"JCQA 番号の確率 {shot}"] = preds[m, shot] == gold
        outputs[m, shot] = generate(texts, 16)
        per_item[m, f"JCQA 生成 {shot}"] = np.array([parse_number(o) for o in outputs[m, shot]]) == gold
    for k in [0, 2]:
        outputs[m, "qa", k] = generate([chat(qa_prompt(ex), qa_shots[:k]) for ex in jsquad], 32)
        per_item[m, f"JSQuAD 完全一致 {k}-shot"] = np.array([float(any(normalize(o) == normalize(g) for g in ex["answers"]["text"]))
                                                         for o, ex in zip(outputs[m, "qa", k], jsquad)])
        per_item[m, f"JSQuAD 文字 F1 {k}-shot"] = np.array([char_f1(o, ex["answers"]["text"]) for o, ex in zip(outputs[m, "qa", k], jsquad)])
    print(f"{m}: {time.time() - t:.0f} 秒, GPU メモリ最大 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    del model; gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

sarashina2.2-0.5b: 63 秒, GPU メモリ最大 3.8 GB
sarashina2.2-1b: 66 秒, GPU メモリ最大 5.9 GB
sarashina2.2-3b: 128 秒, GPU メモリ最大 12.0 GB


In [5]:
# 得点 (%) と、ブートストラップによる 95% 信頼区間
rng = np.random.default_rng(0)
def ci(x, n_boot=2000):
    idx = rng.integers(0, len(x), (n_boot, len(x)))
    lo, hi = np.percentile(x[idx].mean(axis=1), [2.5, 97.5]) * 100
    return f"{x.mean() * 100:5.1f} [{lo:4.1f}, {hi:4.1f}]"

table = pd.Series({k: ci(v) for k, v in per_item.items()}).unstack(0)
models = [n.split("/")[-1].replace("-instruct-v0.1", "") for n in MODELS]
pd.set_option("display.width", 200)
print(table.loc[list(dict.fromkeys(k[1] for k in per_item)), models].to_string())
print(f"(JCQA {len(jcqa)} 問、偶然の正解率 20%。JSQuAD {len(jsquad)} 問)")

                                sarashina2.2-0.5b     sarashina2.2-1b     sarashina2.2-3b
JCQA 選択肢の尤度 (合計)                60.4 [56.4, 64.8]   66.6 [62.4, 70.6]   69.2 [65.2, 73.2]
JCQA 選択肢の尤度 (トークン平均)            48.6 [44.2, 52.8]   55.6 [51.4, 59.8]   52.8 [48.4, 57.2]
JCQA 番号の確率 0-shot               72.4 [68.4, 76.2]   84.8 [81.6, 88.0]   90.4 [87.8, 93.0]
JCQA 生成 0-shot                  52.6 [48.2, 57.2]   81.6 [78.2, 84.8]   90.4 [87.6, 92.8]
JCQA 番号の確率 3-shot (正解が偏った例)     56.4 [52.2, 60.8]   81.6 [78.2, 85.0]   91.0 [88.4, 93.4]
JCQA 生成 3-shot (正解が偏った例)        56.2 [51.8, 60.8]   81.6 [78.2, 85.0]   91.0 [88.4, 93.4]
JCQA 番号の確率 3-shot (正解が重ならない例)   62.4 [58.2, 66.6]   82.2 [78.8, 85.4]   89.8 [87.0, 92.4]
JCQA 生成 3-shot (正解が重ならない例)      62.6 [58.4, 66.8]   82.2 [78.8, 85.4]   89.8 [87.0, 92.4]
JSQuAD 完全一致 0-shot               0.0 [ 0.0,  0.0]    0.5 [ 0.0,  1.5]    9.5 [ 6.0, 13.5]
JSQuAD 文字 F1 0-shot             22.0 [20.1, 24.0]   23.7 [21.5, 26.0]   36.8 [33.3, 40.5]
JSQuAD 完全一

In [6]:
# 番号の確率で選んだ答えの分布 (評価データの正解は 1〜5 にほぼ均等) と、生成方式で番号を取り出せなかった割合
for m in models:
    print(m, {shot: sorted(Counter(int(p) + 1 for p in preds[m, shot]).items()) for shot in SHOTS})
    print("   番号を取り出せなかった割合:", {shot: f"{np.mean([parse_number(o) < 0 for o in outputs[m, shot]]):.1%}" for shot in SHOTS})
print(list(outputs[models[0], "0-shot"][:5]))
print(list(outputs[models[0], "qa", 0][:5]), "/ 正解:", [ex["answers"]["text"][0] for ex in jsquad.select(range(5))])

sarashina2.2-0.5b {'0-shot': [(1, 47), (2, 111), (3, 107), (4, 159), (5, 76)], '3-shot (正解が偏った例)': [(1, 11), (2, 70), (3, 73), (4, 225), (5, 121)], '3-shot (正解が重ならない例)': [(1, 20), (2, 103), (3, 100), (4, 239), (5, 38)]}
   番号を取り出せなかった割合: {'0-shot': '28.8%', '3-shot (正解が偏った例)': '1.0%', '3-shot (正解が重ならない例)': '0.0%'}
sarashina2.2-1b {'0-shot': [(1, 79), (2, 112), (3, 100), (4, 122), (5, 87)], '3-shot (正解が偏った例)': [(1, 65), (2, 113), (3, 93), (4, 121), (5, 108)], '3-shot (正解が重ならない例)': [(1, 72), (2, 143), (3, 103), (4, 131), (5, 51)]}
   番号を取り出せなかった割合: {'0-shot': '6.8%', '3-shot (正解が偏った例)': '0.0%', '3-shot (正解が重ならない例)': '0.0%'}
sarashina2.2-3b {'0-shot': [(1, 95), (2, 122), (3, 108), (4, 105), (5, 70)], '3-shot (正解が偏った例)': [(1, 94), (2, 120), (3, 111), (4, 105), (5, 70)], '3-shot (正解が重ならない例)': [(1, 94), (2, 123), (3, 107), (4, 111), (5, 65)]}
   番号を取り出せなかった割合: {'0-shot': '0.2%', '3-shot (正解が偏った例)': '0.0%', '3-shot (正解が重ならない例)': '0.0%'}
['福井県の南西部、若狭のほぼ中央に位置する市は「小浜」です。', 'セブンイレブンやローソンといえば、どんなお

In [7]:
# モデルの差は偶然の範囲か: 同じ問題で比べる「対応のある」ブートストラップで、差の 95% 信頼区間を求める
def diff_ci(a, b, n_boot=2000):
    idx = rng.integers(0, len(a), (n_boot, len(a)))
    d = a[idx].mean(axis=1) - b[idx].mean(axis=1)
    return f"{(a.mean() - b.mean()) * 100:+5.1f} [{np.percentile(d, 2.5) * 100:+5.1f}, {np.percentile(d, 97.5) * 100:+5.1f}]"

for setting in ["JCQA 番号の確率 0-shot", "JCQA 生成 0-shot", "JSQuAD 文字 F1 0-shot"]:
    print(setting, "| 1b − 0.5b:", diff_ci(per_item[models[1], setting], per_item[models[0], setting]),
          "| 3b − 1b:", diff_ci(per_item[models[2], setting], per_item[models[1], setting]))

# few-shot の効果も同じ方法で確かめる (3-shot − 0-shot)
for method in ["番号の確率", "生成"]:
    for shot in list(SHOTS)[1:]:
        print(f"{method} {shot}:", {m: diff_ci(per_item[m, f"JCQA {method} {shot}"], per_item[m, f"JCQA {method} 0-shot"]) for m in models})

# 評価の件数を減らすと、信頼区間はどれくらい広がるか (3b、番号の確率、0-shot)
x = per_item[models[2], "JCQA 番号の確率 0-shot"]
for n in [50, 100, 200, 500]:
    print(f"{n:3d} 問: {ci(x[:n])}")

JCQA 番号の確率 0-shot | 1b − 0.5b: +12.4 [ +8.6, +16.4] | 3b − 1b:  +5.6 [ +2.2,  +9.0]
JCQA 生成 0-shot | 1b − 0.5b: +29.0 [+24.2, +34.0] | 3b − 1b:  +8.8 [ +5.2, +12.2]
JSQuAD 文字 F1 0-shot | 1b − 0.5b:  +1.7 [ +0.1,  +3.3] | 3b − 1b: +13.1 [ +9.6, +16.6]
番号の確率 3-shot (正解が偏った例): {'sarashina2.2-0.5b': '-16.0 [-20.0, -11.8]', 'sarashina2.2-1b': ' -3.2 [ -6.4,  +0.0]', 'sarashina2.2-3b': ' +0.6 [ -1.8,  +2.8]'}
番号の確率 3-shot (正解が重ならない例): {'sarashina2.2-0.5b': '-10.0 [-13.6,  -6.4]', 'sarashina2.2-1b': ' -2.6 [ -6.2,  +1.2]', 'sarashina2.2-3b': ' -0.6 [ -3.0,  +1.6]'}
生成 3-shot (正解が偏った例): {'sarashina2.2-0.5b': ' +3.6 [ -2.2,  +9.2]', 'sarashina2.2-1b': ' +0.0 [ -3.6,  +3.4]', 'sarashina2.2-3b': ' +0.6 [ -1.6,  +3.0]'}
生成 3-shot (正解が重ならない例): {'sarashina2.2-0.5b': '+10.0 [ +4.8, +15.4]', 'sarashina2.2-1b': ' +0.6 [ -3.2,  +4.4]', 'sarashina2.2-3b': ' -0.6 [ -3.0,  +1.6]'}
 50 問:  88.0 [78.0, 96.0]
100 問:  88.0 [81.0, 94.0]
200 問:  87.0 [82.5, 91.5]
500 問:  90.4 [87.8, 93.0]


- 同じモデルでも、答えの取り出し方と few-shot の例で点数が大きく変わる（3b の JCommonsenseQA で 52.8〜91.0%）。指示文の言い回しでも変わる（この章の「指示文の言語を変えて LLM の出力がどう変わるか比べる」）。モデルはすべての設定をそろえて比べ、設定を明記して報告する。設定の違う論文やリーダーボードの数値と並べてはいけない。
- 選択肢の尤度の方式だけは、選択肢を並べずに質問だけを見せる。ここでは 60.4〜69.2% と、番号を答えさせる方式より低かった。この章の「選択肢の尤度を比べて LLM に分類させる」は長さで割ることを勧めているが、ここではトークン平均にすると下がり、1b (55.6%) と 3b (52.8%) の順序も入れ替わった（信頼区間は重なる）。効くかどうかはデータによるので、両方を確かめる。指示に従う力が要らないので事前学習だけのモデルにも使えるが、その場合はチャットテンプレートを使わず、素のテキストのプロンプトにする。
- 生成させて答えを取り出す方式では、点数に「形式に従う力」が混ざる。0.5b は 0-shot で 28.8% の出力から番号を取り出せず、番号の確率で選ぶと 72.4% のところが 52.6% だった。3-shot で形式を見せると取り出せない出力は 1.0% 以下になり、両方式の正解率の差は 3 モデルとも 0.2 ポイント以内になった。最初に出た数字を答えとみなす取り出し方は単純で、数字を含む問題文を繰り返した出力では誤った数字を拾いうる。
- few-shot の例は答えの分布を動かす。0.5b は 0-shot の時点で「4」に偏っており（500 問中 159 回。正解は 101 問）、3-shot ではさらに「4」が増えて（225 回、239 回）、番号の確率の正解率は例の正解が「4, 5, 5」のとき −16.0 ポイント [−20.0, −11.8]、「4, 5, 1」のとき −10.0 ポイント [−13.6, −6.4] 下がった。「5」を 2 回含む例では「5」の予測が 76 回から 121 回に増えた（多数派のラベルに引っ張られる現象。この章の「Few-shot プロンプトでテキストを分類する」も参照）。ただし正解が重ならない例でも下がったので、偏りだけが原因ではない。3b ではどちらの例でも信頼区間が 0 をまたいだ。点数だけでなく予測の分布も見る。
- JSQuAD を 0-shot で生成させると「〜です。」のような文で答え、完全一致は 9.5% 以下だった。2-shot で 3b は完全一致 28.0%、文字 F1 49.0% になった（指標は質問応答の章の「質問応答を完全一致と F1 で評価する」）。文字 F1 の 1b − 0.5b は +1.7 ポイント [+0.1, +3.3] と小さく、JCommonsenseQA の番号の確率（+12.4 ポイント）とは差の見え方が違う。性質の違うタスクを複数使う。
- 件数が少ないと信頼区間が広い。3b の同じ設定で、50 問では [78.0, 96.0]、500 問では [87.8, 93.0] だった。信頼区間の求め方は評価指標と統計的検定の章の「ブートストラップ法で評価値の信頼区間を求める」を参照。モデルの差は同じ問題で比べる対応のある方法で確かめる（評価指標と統計的検定の章の「2 つのシステムの差が有意かどうかを検定する」）。
- 評価データの汚染: JGLUE は公開されており、sarashina2.2 の事前学習モデルのモデルカードには「Web コーパスから抽出した日本語・英語・コードのデータ」などで学習したとあるだけで、評価データが含まれていないかは確かめられない。含まれていれば点数は実力より高く出る。同じデータセットの学習データで指示チューニングしたモデル（llm-jp-eval は学習データからチューニング用データ `jaster` を作れる）も、同じ形式の問題で有利になりうる。新しく作った問題や非公開の問題でも確かめるとよい。汚染を調べる方法は、この章の「ベンチマークの評価データが学習に漏れていないかを調べる（データ汚染の検出）」を参照。
- 評価の枠組み: EleutherAI の lm-evaluation-harness（MIT ライセンス）には JCommonsenseQA や JSQuAD を含む日本語のタスク群 `japanese_leaderboard` があり、LLM-jp の llm-jp-eval (Apache 2.0) も JGLUE などを横断して評価できる。sarashina2.2 の開発元は、評価ツール flexeval でのチャット評価（MT-Bench など）の設定を公開している。枠組みが違えばプロンプトや答えの取り出し方も違うので、同じ枠組み・同じ設定の数値どうしで比べる。